# Data Analysis using __PySpark__  
*Fun with the __MovieLens__ dataset*  

**Harder Problems, Moar Fun, *with Hints***

<font color='green'>__Support for Google Colab__  </font>

open this notebook in Colab using the following button:  
  
<a href="https://colab.research.google.com/github/shauryashaurya/learn-data-munging/blob/main/03-Spark/003.02-Harder-Problems-in-PySpark(Questions-with-Hints).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>  

<font color='green'>uncomment and execute the cell below to setup and run this Spark notebook on Google Colab.</font>

# Setup, Loading Data, yada yada... 

we've done this before...

## update Jan 2025 
It has been very frustrating to run spark 3.5.x with Python 3.12  
While the official documentation says it works, the python workers for pySpark have been crashing randomly.   
Not cool!  

So for the time being we'll stick with Spark 3.5.4 (ditto for pySpark - v3.5.4) with Python 3.11.xSo for the time being we'll stick with Spark 3.5.4 (ditto for pySpark - v3.5.4) with Python 3.11.x
See [this](https://www.reddit.com/r/dataengineering/comments/1dupogi/wasted_45_hours_to_install_pyspark_locally_pain/) reddit post that shares my frustration.  

In [ ]:
# # SETUP FOR COLAB: select all the lines below and uncomment (CTRL+/ on windows)

# # grab spark
# # as of Jan 2025, the *working* version is 3.5.4, get the link from Apache Spark's website
# ! wget -q https://dlcdn.apache.org/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz
# # unzip spark
# !tar xf spark-3.5.4-bin-hadoop3.tgz
# # install findspark package
# !pip install -q findspark
# # Let's download and unzip the MovieLens 25M Dataset as well.
# ! mkdir ./../data
# ! wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip
# ! unzip ./ml-25m.zip -d ./../data/

# # got to provide JAVA_HOME and SPARK_HOME vairables
# import os
# # IMPORTANT - check the version of java, use 11 or 17, code not tested on 21 yet
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
# # IMPORTANT - UPDATE THE SPARK_HOME PATH BASED ON THE PACKAGE YOU DOWNLOAD
# os.environ["SPARK_HOME"] = "/content/spark-3.5.4-bin-hadoop3"
# ! echo "DONE"

**Citation**:  
*F. Maxwell Harper and Joseph A. Konstan.* 2015.  
The MovieLens Datasets: History and Context.  
ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>  

In [ ]:
# import  perf_counter_ns() 
from time import perf_counter_ns 
import math

## Setup the Spark Cluster

In [ ]:
# Step 1: initialize findspark
import findspark

findspark.init()

In [ ]:
# Step 2: import pyspark
import pyspark
from pyspark.sql import SparkSession

pyspark.__version__

In [ ]:
# Step 3: Create a spark session

# using local[*] to use as many logical cores as available, use 1 when in doubt
# 'local[1]' indicates spark on 1 core on the local machine or specify the number of cores needed
# use .config("spark.some.config.option", "some-value") for additional configuration

start = perf_counter_ns()
# 
spark = (
    SparkSession.builder.master("local[1]")
    .appName("Analyzing Movielens Data")
    .getOrCreate()
)
# 
stop = perf_counter_ns()

In [ ]:
print("time it took to start the spark session: ", round(((stop-start)/1000000000),4))

In [ ]:
# spark

## Load data, carefull with the schema

### Schema Spec

Here's the list of files (as of Aug 2022) that you get when you unzip the dataset:
1. **movies**.csv - list of movies with at least one rating.  
    Header: ```movieId,title,genres```  
1. **links**.csv - IDs to generate links to the movie listing on imdb.com and themoviedb.org  
    Header: ```movieId,imdbId,tmdbId```  
1. **ratings**.csv - Each line of this file after the header row represents one rating of one movie by one user.  
    Header: ```userId,movieId,rating,timestamp```  
1. **tags**.csv - Each line of this file after the header row represents one tag applied to one movie by one user.  
    Header: ```userId,movieId,tag,timestamp```  
1. Tag Genome: The tag genome contains tag relevance scores for movies. See [this](http://files.grouplens.org/papers/tag_genome.pdf)  
	1. **genome-tags**.csv - A list of tags  
    Header: ```tagId,tag```  
	1. **genome-scores**.csv - Each movie in the genome has a relevance score value for every tag in the genome  
    Header: ```movieId,tagId,relevance```  
1. README.txt - Check out the README.txt for more details about the files.  

### Data encoding details

From the Readme file, we have the following observations about the data:
1. Each file is a CSV with a single header row
1. Separator char is ```,```
1. Escape char is ```"```
1. Encoding is UTF-8

Let's set these options when reading the CSV files.

### Specify the schema for Spark  
  
Avoid ```inferSchema``` as much as possible, just cleaner

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.rdd import RDD


In [ ]:
#
schema_movies = StructType(
    [
        StructField("movieId", StringType(), False),
        StructField("title", StringType(), False),
        StructField("genres", StringType(), True),
    ]
)

In [ ]:
#
schema_links = StructType(
    [
        StructField("movieId", StringType(), False),
        StructField("imdbId", StringType(), True),
        StructField("tmdbId", StringType(), True),
    ]
)

In [ ]:
#
schema_ratings = StructType(
    [
        StructField("userId", StringType(), False),
        StructField("movieId", StringType(), False),
        StructField("rating", FloatType(), True),
        StructField("timestamp", StringType(), True),
    ]
)

In [ ]:
#
schema_tags = StructType(
    [
        StructField("userId", StringType(), False),
        StructField("movieId", StringType(), False),
        StructField("tag", StringType(), True),
        StructField("timestamp", StringType(), True),
    ]
)

In [ ]:
#
schema_genome_tags = StructType(
    [
		StructField("tagId", StringType(), False), 
		StructField("tag", StringType(), False)
	]
)

In [ ]:
#
# using arbitrary precision signed decimals (java.math.BigDecimal) for relevance scores
schema_genome_scores = StructType(
    [
        StructField("movieId", StringType(), False),
        StructField("tagId", StringType(), False),
        StructField("relevance", DecimalType(), False),
    ]
)

### Specify the location of your data  

Change this folder if you are saving the data at the different place

In [ ]:
datalocation = "../data/ml-25m/"

In [ ]:
# specify file names
file_path_movies = datalocation + "movies.csv"
file_path_links = datalocation + "links.csv"
file_path_ratings = datalocation + "ratings.csv"
file_path_tags = datalocation + "tags.csv"
file_path_genome_tags = datalocation + "genome-tags.csv"
file_path_genome_scores = datalocation + "genome-scores.csv"

### Load the data and review

Let's load each file in turn and observe, just to get a sense of familiarity with the data.  

#### Movies

In [ ]:
movies_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_movies)
    .load(file_path_movies)
)

In [ ]:
# Spark collects all transformations needed
# and execution doesn't begin until an "action" is triggered
# 
# 'show' triggers a partial execution 
#  'show' - limiting computation (where relevant) to the number of rows you want to display
movies_raw.show(10, False)

#### Links

In [ ]:
links_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_links)
    .load(file_path_links)
)

In [ ]:
links_raw.show(10, False)

#### Ratings

In [ ]:
ratings_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_ratings)
    .load(file_path_ratings)
)

In [ ]:
ratings_raw.show(10, False)

#### Tags

In [ ]:
tags_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_tags)
    .load(file_path_tags)
)

In [ ]:
tags_raw.show(10, False)

#### Tag Genome

In [ ]:
genome_tags_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_genome_tags)
    .load(file_path_genome_tags)
)

In [ ]:
genome_tags_raw.show(10, False)

#### Tag Genome Scores

In [ ]:
genome_scores_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_genome_scores)
    .load(file_path_genome_scores)
)

In [ ]:
genome_scores_raw.show(10, False)

# Harder problems on `MovieLens` using `PySpark`
  
Look, if we've done the previous notebooks (002.xx) many of these would feel a continuation of the same theme or a repetition even.  
In many cases these should really not feel "hard".   
  
I have put these together to say: "here's all the ones that appear hard, also some patterns that I have seen across them"

## **Patterns** I have frequently seen in some of the "harder"-ish-real-world-adjacent problems

### **A. Figure out a Sequence or do some Pattern Detection** 
analyzing sequences of events/facts, not just individual events or simple aggregations.   
This demands more sophisticated use of window functions, date/time manipulations, and potentially iterative processing. Iterative nearly always blows up in production. :)  

### **B. Implicit Graph-Like Relationships** 
There are situations where you are dealing with an implicit Graph-Like structure (hierarchies, relationships across entities, business rules, reachability etc. etc.).   
This is where you've got to start thinking about how to represent and traverse relationships that are not explicitly defined as foreign keys.  

### **C. Complex Windowing and Grouping** 
combinations of window functions, aggregations, and filtering...  
just the perfect recipe for a drawn-out struggle with your spark jobs, late night on a Saturday as your family slowly forgets what you look like and your existence hurtles towards decrepitude. Just chef's kiss.   
The fix? A deep (deeper?) understanding of how these operations interact, review early, warn often.

### **D. Transitive Closure** 
Fancy word, (just to prove I am not a philistine) but really means that if A is connected to B and B is connected to C, then A and C may have a connection.   
This sort of stuff is useful for in all kinds of queries, esp. in a business situation, when it's like 5PM on a Friday and you have plans for the weekend.   
No seriously, T.C. adds a significant layer of complexity that is difficult to handle efficiently in a distributed environment like Spark.    
Think of transitive closure like **friend-of-a-friend** situation on a social network.    
    
- Suppose **Alice knows Bob** and **Bob knows Charlie**.    
- Even though Alice and Charlie don’t directly know each other, we can *infer* that Alice can be connected to Charlie through Bob.    
    
In a **transitive closure**, we keep expanding these connections until we capture all possible indirect relationships, or your spark cluster has burned down (like obvs...) or Chapter 11 rumours start going around due to all the cloud costs.     
Here's some common scenarios:     
- **Flights between cities**: If you can fly from **New York → London** and from **London → Paris**, then logically, you can reach **New York → Paris** via London.    
- **Supply Chains**: If a **raw material supplier** delivers to **Factory A**, and Factory A supplies to **Warehouse B**, then the raw material can reach **Warehouse B** indirectly.    
     
Or in the case of our `MovieLens` dataset context, let’s say two users have *similar taste* if they rate the same movies (that same friend-of-a-friend idea).     
- **User A rates Movie X**    
- **User B also rates Movie X** → So User A and B are connected.    
- **User B rates Movie Y** → This means User A could also be connected to Movie Y (indirectly).    
       
Use **transitive closure** to expand these connections, so we can **recommend movies based on indirect user interactions**.    
Other interesting queries come to mind:   
- **Find Hidden Similar Users**: If User A indirectly connects to User C via B, see we can recommend movies watched by C to A.     
- **Detect User Communities**: find groups of users with similar movie preferences based on extended relationships.     

### **E. Non-Standard Aggregations**  
Stuff like "average path length" and "longest consecutive sequence" type calculations are not standard aggregations.   
They require custom logic within the Spark framework and hair, lots of hair on your head, so you can pull some out while solving for these...    

### **Optimization Techniques** I have often used in these cases (esp. our friend the Transitive Closure)
* Use **broadcast joins** for small subsets - Instead of costly shuffles, we **broadcast** one side of the join when the dataset is small.  
* Use **temporary tables** for better query performance - yeah right (snigger)    
* Leverage **SQL recursion** type approach for efficiency - this is iffy, use as a last resort, like all your bullets are spent, your knife blade is rusty, there's a pack of lions at the door, your mushrooms just kicked in and there's not much else you can do. 

## Question Set

For each question below, I provide a 'theme' in brackets - helps one think of the possible approaches and classify the 'kind' of query or analysis we are trying to crack.

### **1.  Transitive Genre Preference Propagation (Graph-like + Optimization)**    
    
---    
    
#### Question    
       
        
*   A user directly rates movies.
*   We define *indirect* genre preference as follows:
    *   If User A rates 5 movies of Genre X highly (>=4 stars),
    *   and User A also rates 5 movies of Genre Y highly,
    *   then all users who rated at least 3 movies of Genre Y highly *also* indirectly prefer Genre X.     
.          
*   Propagate this preference transitively:
    *   **Output the top 5 *indirectly* preferred genres for each user that are NOT directly preferred.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Combines transitive closure (graph-like propagation) with filtering based on direct preferences.  Requires iterative processing and careful handling of already-propagated preferences to avoid infinite loops.  The "indirect, not direct" condition is crucial and easily missed.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Consider using a combination of iterative DataFrame joins and filtering, with careful use of caching and broadcasting smaller DataFrames (like direct genre preferences per user) to minimize shuffle operations.  Explore accumulator usage for tracking convergence. Explore converting to RDDs for finer-grained control over partitioning if DataFrames become a bottleneck.</font>         
        

**Using DataFrames**

In [ ]:
# --- Helper Functions (you've seen this before, chill...) ---

def expand_genres(movies_df):
    # Expands the pipe-separated genres into individual rows...
    return movies_df.withColumn("genre", F.explode(F.split(F.col("genres"), "\|")))


In [ ]:
ratings = ratings_raw
movies = movies_raw
exec_times = []

In [ ]:
# 1. Filter high ratings (>= 4)
high_ratings = ratings.filter(F.col("rating") >= 4.0)

In [ ]:
# 2. Join with movies and expand genres
movies_expanded = expand_genres(movies)
user_genre_ratings = high_ratings.join(movies_expanded, "movieId")

In [ ]:
user_genre_ratings.show(5)

According to the problem:

If User A rates 5+ movies of Genre X highly AND 5+ movies of Genre Y highly  
Then all users who rated 3+ movies of Genre Y highly also indirectly prefer Genre X  

In [ ]:
# 3.1 Calculate direct preferences for different thresholds
# Users who rated at least 5 movies of a genre highly
direct_prefs_5plus = user_genre_ratings.groupBy("userId", "genre") \
    .agg(F.count("*").alias("rating_count")) \
    .filter(F.col("rating_count") >= 5)

In [ ]:
# measure time, to see how long this takes...
# next to show() as it'll trigger the evaluation
start = perf_counter_ns()
# 

direct_prefs_5plus.show(5)
# 
stop = perf_counter_ns()

Checking elapsed time in this manner is not exactly scientific, it changes from system to system and based on spark configuration, but it does give an intution on which jobs took time. 

In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 3.1 took ",elapsed," seconds")
exec_times.append(("Step 3.1",elapsed))

In [ ]:
# 3.2 Users who rated at least 3 movies of a genre highly (for indirect preferences)
direct_prefs_3plus = user_genre_ratings.groupBy("userId", "genre") \
    .agg(F.count("*").alias("rating_count")) \
    .filter(F.col("rating_count") >= 3)

In [ ]:
start = perf_counter_ns()
# 

direct_prefs_3plus.show(5)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 3.2 took ",elapsed," seconds")
exec_times.append(("Step 3.2",elapsed))

In [ ]:
# 4. Establish genre-to-genre relationships
# If a user directly prefers both Genre X and Genre Y, then liking Genre Y implies liking Genre X
genre_relationships = direct_prefs_5plus.alias("a").join(
    direct_prefs_5plus.alias("b"),
    "userId"
    ).filter(F.col("a.genre") != F.col("b.genre")) \
     .select(
         F.col("b.genre").alias("fromGenre"),  # Genre Y
         F.col("a.genre").alias("toGenre")     # Genre X
    ).distinct()

In [ ]:
start = perf_counter_ns()
# 

genre_relationships.show(10)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 4 took ",elapsed," seconds")
exec_times.append(("Step 4",elapsed))

In [ ]:
genre_relationships.count()

In [ ]:
genre_relationships.schema

In [ ]:
# 5. Propagate genre relationships transitively
# If Genre Y implies Genre X and Genre Z implies Genre Y, then Genre Z implies Genre X
extended_relationships = genre_relationships
for i in range(2):  # Limited iterations for the transitive closure
    new_relationships = extended_relationships.alias("a").join(
        genre_relationships.alias("b"),
        F.col("a.toGenre") == F.col("b.fromGenre")
        ).select(
            F.col("a.fromGenre").alias("fromGenre"),
            F.col("b.toGenre").alias("toGenre")
        ).distinct()
    
    extended_relationships = extended_relationships.union(new_relationships).distinct()


In [ ]:
start = perf_counter_ns()
# 

extended_relationships.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 5 took ",elapsed," seconds")
exec_times.append(("Step 5",elapsed))

In [ ]:
extended_relationships.count()

In [ ]:
# 6. Calculate indirect preferences
# If a user rates 3+ movies of Genre Y highly and Genre Y implies Genre X, 
# then the user indirectly prefers Genre X
indirect_prefs = direct_prefs_3plus.alias("u").join(
    extended_relationships.alias("r"),
    F.col("u.genre") == F.col("r.fromGenre")
    ).select(
        F.col("u.userId").alias("userId"),
        F.col("r.toGenre").alias("indirectGenre")
    ).distinct()

In [ ]:
start = perf_counter_ns()
# 

indirect_prefs.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 6 took ",elapsed," seconds")
exec_times.append(("Step 6",elapsed))

In [ ]:
# 7.1 check ALL direct preferences
all_direct_prefs = user_genre_ratings.groupBy("userId", "genre").count() \
    .select(F.col("userId"), F.col("genre").alias("directGenre"))

In [ ]:
start = perf_counter_ns()
# 

all_direct_prefs.show(5)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 7.1 took ",elapsed," seconds")
exec_times.append(("Step 7.1",elapsed))

In [ ]:
# 7.2 Remove preferences that are already direct - need to check ALL direct preferences
indirect_prefs_filtered = indirect_prefs.join(
    all_direct_prefs,
    (indirect_prefs.userId == all_direct_prefs.userId) & 
    (indirect_prefs.indirectGenre == all_direct_prefs.directGenre),
    "left_anti"  # Only keep rows that don't match (not directly preferred)
)

In [ ]:
start = perf_counter_ns()
# 

indirect_prefs_filtered.show(5)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 7.2 took ",elapsed," seconds")
exec_times.append(("Step 7.2",elapsed))

In [ ]:
# 8. Count how many ways each user indirectly prefers each genre
indirect_pref_counts = indirect_prefs_filtered.groupBy("userId", "indirectGenre") \
    .agg(F.count("*").alias("count"))

In [ ]:
start = perf_counter_ns()
# 

indirect_pref_counts.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 8 took ",elapsed," seconds")
exec_times.append(("Step 8",elapsed))

In [ ]:
# 9. Get top 5 indirect preferences for each user
windowSpec = Window.partitionBy("userId").orderBy(F.desc("count"))
top_indirect_prefs = indirect_pref_counts \
    .withColumn("rank", F.row_number().over(windowSpec)) \
    .filter(F.col("rank") <= 5) \
    .select("userId", F.col("indirectGenre").alias("genre"))

In [ ]:
start = perf_counter_ns()
# 

top_indirect_prefs.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 9 took ",elapsed," seconds")
exec_times.append(("Step 9",elapsed))

In [ ]:
import plotly.express as px

In [ ]:
# convert each part of exec_times to a simple list for plotly
fig_exec_times = px.bar(exec_times, x=[val[0] for val in exec_times], y=[val[1] for val in exec_times])
fig_exec_times.show()

:)

#### Optimize the solution


In [ ]:
# clear cache
spark.catalog.clearCache()

The *Adaptive Query Engine* in Spark is by default enabled for Spark v3.2 and later.

In [ ]:
# this will impact the rest of the notebook, so be careful if you want to note unoptimized times
# Memory configuration
# spark.conf.set("spark.driver.memory", "4g")
# spark.conf.set("spark.executor.memory", "18g")

# Limit memory fraction to avoid OOM errors
# spark.conf.set("spark.memory.fraction", "0.8")

# Enable offheap memory if processing large datasets
# spark.conf.set("spark.memory.offHeap.enabled", "true")
# spark.conf.set("spark.memory.offHeap.size", "4g")

# Enable adaptive query execution
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

In [ ]:
ratings = ratings_raw
movies = movies_raw
opt_exec_times = []

**Optimization: use `cache`**

In [ ]:
# 1. Filter high ratings (>= 4) - CACHED
high_ratings = ratings.filter(F.col("rating") >= 4.0).cache()

**Optimization: `cache` with `broadcast`**

In [ ]:
# 2. Join with movies and expand genres - CACHED with BROADCAST
movies_expanded = expand_genres(movies)
user_genre_ratings = high_ratings.join(
    F.broadcast(movies_expanded), 
    "movieId"
).cache()

In [ ]:
user_genre_ratings.show(5)

**Optimization: use `cache`**

In [ ]:
# 3.1 Calculate direct preferences for different thresholds
# Users who rated at least 5 movies of a genre highly
direct_prefs_5plus = user_genre_ratings.groupBy("userId", "genre") \
    .agg(F.count("*").alias("rating_count")) \
    .filter(F.col("rating_count") >= 5) \
    .cache()

In [ ]:
# measure time, to see how long this takes...
# next to show() as it'll trigger the evaluation
start = perf_counter_ns()
# 

direct_prefs_5plus.show(5)
# 
stop = perf_counter_ns()

In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 3.1 took ",elapsed," seconds")
opt_exec_times.append(("Step 3.1",elapsed))

**Optimization: use `cache`**

In [ ]:
# 3.2 Users who rated at least 3 movies of a genre highly (for indirect preferences)
direct_prefs_3plus = user_genre_ratings.groupBy("userId", "genre") \
    .agg(F.count("*").alias("rating_count")) \
    .filter(F.col("rating_count") >= 3) \
    .cache()

In [ ]:
start = perf_counter_ns()
# 

direct_prefs_3plus.show(5)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 3.2 took ",elapsed," seconds")
opt_exec_times.append(("Step 3.2",elapsed))

**Optimization: Partitioning for Better Distribution**  
You can see below that we start by performing partitioning and cacheing.  
However, I am running this code on a laptop with 4 cores. So these prove to be an overhead for my system.  
Accordingly, have commented out both cache() and repartition().  
This is a very common exercise you'll have to perform based on the size and configuration of the cluster.

In [ ]:
# 4. Establish genre-to-genre relationships - REPARTITIONED + CACHED
# If a user directly prefers both Genre X and Genre Y, then liking Genre Y implies liking Genre X
# YMMV - this is based on the configuration of my machine, change it based on the cluster your are running...
genre_relationships = direct_prefs_5plus.alias("a").join(
    direct_prefs_5plus.alias("b"),
    "userId"
).filter(F.col("a.genre") != F.col("b.genre")) \
 .select(
     F.col("b.genre").alias("fromGenre"),  # Genre Y
     F.col("a.genre").alias("toGenre")     # Genre X
 ).distinct() #\
# cache materialization seems to be taking this too 770+ seconds! so disabling cache here.
# .cache()
# tried repartition, it took 32 to 40x the time, I am using a single laptop, 
# in my case repartitioning may not be the best idea...
# .repartition(28) \

In [ ]:
start = perf_counter_ns()
# 

genre_relationships.show(10)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 4 took ",elapsed," seconds")
opt_exec_times.append(("Step 4",elapsed))

In [ ]:
genre_relationships.count()

In [ ]:
genre_relationships.schema

**Optimization: Optimized transitive closure algorithm with _delta tracking_**

In [ ]:
def compute_transitive_closure(edges_df, max_iterations=3):
    """Optimized transitive closure computation with delta approach"""
    
    # Initial set of edges
    tc_df = edges_df.cache()
    
    # Track delta (new edges) for each iteration
    delta_df = tc_df
    
    for i in range(max_iterations):
        # Find newly reachable edges by combining delta with all edges
        new_edges = delta_df.alias("delta").join(
            tc_df.alias("tc"),
            F.col("delta.toGenre") == F.col("tc.fromGenre")
        ).select(
            F.col("delta.fromGenre"),
            F.col("tc.toGenre")
        ).distinct()
        
        # The issue is here - we need proper aliases in the join condition
        # Only keep edges we haven't seen before
        delta_df = new_edges.alias("new").join(
            tc_df.alias("existing"),
            (F.col("new.fromGenre") == F.col("existing.fromGenre")) & 
            (F.col("new.toGenre") == F.col("existing.toGenre")),
            "leftanti"
        ).cache()
        
        # If no new edges found, we're done
        if delta_df.count() == 0:
            break
            
        # Add new edges to transitive closure
        tc_df = tc_df.union(delta_df).cache()
    
    return tc_df
# # Optimized transitive closure algorithm with delta tracking
# def compute_transitive_closure(edges_df, max_iterations=3):
#     # Initial set of edges
#     tc_df = edges_df.cache()
    
#     # Track delta (new edges) for each iteration
#     delta_df = tc_df
    
#     # Persist iteration count to allow early termination
#     for i in range(max_iterations):
#         # Find newly reachable edges by combining delta with all edges
#         new_edges = delta_df.alias("delta").join(
#             tc_df.alias("tc"),
#             F.col("delta.toGenre") == F.col("tc.fromGenre")
#         ).select(
#             F.col("delta.fromGenre"),
#             F.col("tc.toGenre")
#         ).distinct()
        
#         # Only keep edges we haven't seen before
#         delta_df = new_edges.join(
#             tc_df,
#             (new_edges.fromGenre == tc_df.fromGenre) & 
#             (new_edges.toGenre == tc_df.toGenre),
#             "leftanti"
#         ).cache()
        
#         # If no new edges found, we're done
#         if delta_df.count() == 0:
#             break
            
#         # Add new edges to transitive closure
#         tc_df = tc_df.union(delta_df).cache()
    
#     return tc_df

In [ ]:
# 5. Propagate genre relationships transitively with OPTIMIZED algorithm
extended_relationships = compute_transitive_closure(genre_relationships, max_iterations=3).cache()

# for comparision
# INITIAL APPROACH
# 5. Propagate genre relationships transitively
# If Genre Y implies Genre X and Genre Z implies Genre Y, then Genre Z implies Genre X
'''
extended_relationships = genre_relationships
for i in range(2):  # Limited iterations for the transitive closure
    new_relationships = extended_relationships.alias("a").join(
        genre_relationships.alias("b"),
        F.col("a.toGenre") == F.col("b.fromGenre")
        ).select(
            F.col("a.fromGenre").alias("fromGenre"),
            F.col("b.toGenre").alias("toGenre")
        ).distinct()
    
    extended_relationships = extended_relationships.union(new_relationships).distinct()
'''


In [ ]:
start = perf_counter_ns()
# 

extended_relationships.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 5 took ",elapsed," seconds")
opt_exec_times.append(("Step 5",elapsed))

In [ ]:
extended_relationships.count()

**Optimization: Optimize `JOIN` using `Broadcast`**  
  
Like for step 4, we start by cache() and broadcasting the smaller dataset.  
This would work on a sufficiently large cluster, but on my 4 core laptop, this is overhead, so we'll have to disable both.  
Your Milage May Vary - you will have to review the cluster capabilities and figure out the optimal configuration.

In [ ]:
# 6. Calculate indirect preferences - OPTIMIZED JOIN
# If a user rates 3+ movies of Genre Y highly and Genre Y implies Genre X, 
# then the user indirectly prefers Genre X
# indirect_prefs = direct_prefs_3plus.alias("u").join(
#     F.broadcast(extended_relationships.alias("r")),  # Broadcast smaller dataset
#     F.col("u.genre") == F.col("r.fromGenre")
#     ).select(
#         F.col("u.userId").alias("userId"),
#         F.col("r.toGenre").alias("indirectGenre")
#     ).distinct().cache()
indirect_prefs = direct_prefs_3plus.alias("u").join(
    extended_relationships.alias("r"),  # for a single laptop, broadcasting may actually slow things down, uncomment the block above if you are using a larger cluster...
    F.col("u.genre") == F.col("r.fromGenre")
    ).select(
        F.col("u.userId").alias("userId"),
        F.col("r.toGenre").alias("indirectGenre")
    ).distinct()#.cache() #disabling cache.


# for comparision
# INITIAL APPROACH
# 6. Calculate indirect preferences
# If a user rates 3+ movies of Genre Y highly and Genre Y implies Genre X, 
# then the user indirectly prefers Genre X
'''
indirect_prefs = direct_prefs_3plus.alias("u").join(
    extended_relationships.alias("r"), #NO BROADCASE HERE, EASY TO MISS
    F.col("u.genre") == F.col("r.fromGenre")
    ).select(
        F.col("u.userId").alias("userId"),
        F.col("r.toGenre").alias("indirectGenre")
    ).distinct()
'''

In [ ]:
start = perf_counter_ns()
# 

indirect_prefs.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 6 took ",elapsed," seconds")
opt_exec_times.append(("Step 6",elapsed))

**Optimization:   
   7.1: `cache()`  
   7.2: Repartition and use `antijoin`**

In [ ]:
# 7.1 Remove preferences that are already direct - IMPROVED ANTI-JOIN for 7.2
# Use more efficient anti-join pattern
all_direct_prefs = user_genre_ratings.groupBy("userId", "genre").count() \
    .select(
        F.col("userId"),
        F.col("genre").alias("directGenre")
    ).cache()

# for comparision
# INITIAL APPROACH
# 7.1 check ALL direct preferences
'''
all_direct_prefs = user_genre_ratings.groupBy("userId", "genre").count() \
    .select(F.col("userId"), F.col("genre").alias("directGenre"))
'''

In [ ]:
start = perf_counter_ns()
# 

all_direct_prefs.show(5)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 7.1 took ",elapsed," seconds")
opt_exec_times.append(("Step 7.1",elapsed))

In [ ]:
# 7.2 Remove preferences that are already direct - IMPROVED ANTI-JOIN for 7.2

# First, repartition indirect_prefs calculated in step 6 before join to improve performance
# repartition on my single laptop (with limited cores) does not optimize, uncomment the line below for a larger cluster...
# indirect_prefs_repartitioned = indirect_prefs.repartition(F.col("userId"))
indirect_prefs_repartitioned = indirect_prefs

# Use more efficient anti-join pattern
indirect_prefs_filtered = indirect_prefs_repartitioned.join(
    all_direct_prefs,
    (indirect_prefs_repartitioned.userId == all_direct_prefs.userId) & 
    (indirect_prefs_repartitioned.indirectGenre == all_direct_prefs.directGenre),
    "leftanti"  # Only keep rows that don't match
)


# for comparision
# INITIAL APPROACH
# 7.2 Remove preferences that are already direct - need to check ALL direct preferences
'''
indirect_prefs_filtered = indirect_prefs.join(
    all_direct_prefs,
    (indirect_prefs.userId == all_direct_prefs.userId) & 
    (indirect_prefs.indirectGenre == all_direct_prefs.directGenre),
    "left_anti"  # Only keep rows that don't match (not directly preferred)
)
'''

In [ ]:
start = perf_counter_ns()
# 

indirect_prefs_filtered.show(5)
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 7.2 took ",elapsed," seconds")
opt_exec_times.append(("Step 7.2",elapsed))

**No Change for #8**

In [ ]:
# 8. Count how many ways each user indirectly prefers each genre
indirect_pref_counts = indirect_prefs_filtered.groupBy("userId", "indirectGenre") \
    .agg(F.count("*").alias("count"))

In [ ]:
start = perf_counter_ns()
# 

indirect_pref_counts.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 8 took ",elapsed," seconds")
opt_exec_times.append(("Step 8",elapsed))

**Optimization: `repartition`**  
Again, while our first optimization approach is to repartition, it leads to performance overhead on my 4 core laptop.  
This would be a strategy for a slightly larger cluster. For now, we'll disable the repartition.

In [ ]:
# 9. Get top 5 indirect preferences for each user - OPTIMIZED WITH PARTITIONING
windowSpec = Window.partitionBy("userId").orderBy(F.desc("count"))

# Calculate and fetch efficiently, not using partitioning due to it causing overheads on my 4 core laptop...
top_indirect_prefs = indirect_pref_counts \
    .withColumn("rank", F.row_number().over(windowSpec)) \
    .filter(F.col("rank") <= 5) \
    .select("userId", F.col("indirectGenre").alias("genre"))

# # Calculate and fetch efficiently with proper partitioning
# top_indirect_prefs = indirect_pref_counts \
#     .repartition(F.col("userId")) \
#     .withColumn("rank", F.row_number().over(windowSpec)) \
#     .filter(F.col("rank") <= 5) \
#     .select("userId", F.col("indirectGenre").alias("genre"))


# for comparision
# INITIAL APPROACH
# 9. Get top 5 indirect preferences for each user
'''
windowSpec = Window.partitionBy("userId").orderBy(F.desc("count"))
top_indirect_prefs = indirect_pref_counts \
    .withColumn("rank", F.row_number().over(windowSpec)) \
    .filter(F.col("rank") <= 5) \
    .select("userId", F.col("indirectGenre").alias("genre"))
'''

In [ ]:
start = perf_counter_ns()
# 

top_indirect_prefs.show()
# 
stop = perf_counter_ns()


In [ ]:
elapsed = round(((stop-start)/1000000000),3)
print("Step 9 took ",elapsed," seconds")
opt_exec_times.append(("Step 9",elapsed))

#### **Comparing the execution time for the initial approach and the optimized version**

In [ ]:
# convert each part of exec_times to a simple list for plotly
fig_exec_times = px.bar(exec_times, x=[val[0] for val in exec_times], y=[val[1] for val in exec_times], title = "Initial Approach")
fig_exec_times.show()

In [ ]:
fig_opt_exec_times = px.bar(opt_exec_times, x=[val[0] for val in opt_exec_times], y=[val[1] for val in opt_exec_times], title = "**Optimized** Approach")
fig_opt_exec_times.show()

Clearly we have some good gains, even if it's on my measly laptop...

### **2.  Time-Decayed Movie Recommendation Similarity (Windowing + Complex-ish Aggregation)**    
    
---    
    
#### Question    
       
        
*   Calculate a time-decayed similarity score between all pairs of movies.
*   The similarity is based on users who have rated both movies.
*   For each user who rated both, the contribution to similarity decays exponentially with the time difference between their ratings of the two movies (half-life of 30 days).
*   **Output the top 10 most similar movies for each movie.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b> Requires joining `ratings` with itself, calculating time differences, applying exponential decay, and then aggregating per movie pair.  The time-decay calculation and aggregation within a window are complex.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Pre-calculate the exponential decay factor based on time difference buckets (e.g., 0-1 day, 1-2 days, etc.) to avoid redundant calculations.  Consider using a broadcast join if one of the movie DataFrames is significantly smaller.  Window functions might seem like a fit, but custom aggregation is likely necessary.</font>         
        

### **3.  Genre Co-occurrence Network Analysis (Graph-like + Aggregation)**    
    
---    
    
#### Question    
       
        
*   **Construct a weighted, undirected graph where nodes are genres, and an edge exists between two genres if at least 100 users have rated at least 5 movies of *each* genre with an average rating of 4 or higher.**
*   The edge weight is the Jaccard similarity of the user sets who contributed to each genre's qualification.
*   **Output the top 3 most "central" genres based on degree centrality (weighted).**      
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires multiple joins and aggregations to determine qualifying genres and users.  Calculating Jaccard similarity efficiently is a challenge.  The concept of "centrality" adds a graph analysis layer.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Use broadcast joins to distribute smaller DataFrames (like user sets per genre).  Consider pre-computing user-genre affinities to speed up subsequent calculations. Explore efficient set intersection and union operations (e.g., using bitsets within a UDF if necessary).</font>         
        

### **4.  Rating Pattern Anomaly Detection (Windowing + Custom Logic)**    
    
---    
    
#### Question    
       
        
*   **Identify users whose rating behavior shows significant *sudden* shifts.**
*   A "shift" is defined as a change in the average rating of at least 1.5 stars (up or down) over a rolling 7-day window, compared to the previous 30-day average, occurring in at least 3 distinct weeks within the dataset.    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Combines rolling window calculations with comparison to a longer-term average.  The "3 distinct weeks" condition adds complexity to avoid false positives.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Use window functions with custom aggregations to calculate both the 7-day and 30-day averages.  Efficient date handling and filtering are crucial. Caching intermediate results is vital.</font>         
        

### **5.  Movie Recommendation Cold Start (Complex Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   For each movie with *fewer* than 50 ratings, 
    *   **identify the top 5 "similar" movies based on a combination of shared genres** (weighted by the number of shared genres) and
    *   **the average rating difference of users who have rated both** (penalize large differences).
*   Handle the case where no users have rated both.    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b> Addresses the cold-start problem, requiring careful handling of missing data and a combination of genre-based and rating-based similarity.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Pre-calculate genre overlaps between movies.  Use coalesce and default values to handle cases with no common raters.  Broadcast join the smaller DataFrame (movies with < 50 ratings).</font>         
        

### **6.  Transitive User Similarity (Graph-like + Optimization)**    
    
---    
    
#### Question    
       
        
*   **Calculate a transitive user similarity score.**
*   If User A and User B have rated at least 5 movies in common with an average rating difference of less than 1, they have a direct similarity.
*   Transitive similarity extends this: If A is similar to B, and B is similar to C, then A and C have a transitive similarity score (decayed by a factor of 0.5 for each hop).
*   **Output the 5 most *transitively* similar users for each user.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Combines transitive closure with a decay factor.  Requires iterative calculation and careful management of similarity scores across hops.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Consider an iterative approach using joins and aggregations, caching intermediate similarity DataFrames.  Use a convergence threshold to stop the iteration. Explore graph libraries (like GraphFrames) if performance is critical, but be aware of the overhead.</font>         
        

### **7.  Time-Based Genre Evolution (Windowing + Complex Aggregation)**    
    
---    
    
#### Question    
       
        
*   **Track the popularity of each genre over time (monthly windows).**
*   Define "popularity" as the average rating of movies of that genre, weighted by the number of ratings in that month.
*   **Identify genres that show a statistically significant (e.g., using a z-score) increase or decrease in popularity over any 6-month period.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires windowing by month, calculating weighted averages, and then applying a statistical test across a longer window.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Pre-calculate monthly aggregations (total ratings, sum of ratings) for each genre.  Use window functions to calculate the z-score efficiently.</font>         
        

### **8.  User Rating Bias Detection (Aggregation + Statistical Analysis)**    
    
---    
    
#### Question    
       
        
*   **Identify users who exhibit a significant rating bias (e.g., consistently rating higher or lower than the average).**
*   Calculate each user's average rating deviation from the movie's average rating, and then identify outliers (e.g., users whose average deviation is more than 2 standard deviations from the mean deviation).    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires calculating per-movie and per-user averages, then combining them to find deviations.  Outlier detection adds a statistical layer.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Use broadcast joins to efficiently combine movie and user statistics.  Calculate the mean and standard deviation of the deviations in a separate step for efficiency.</font>         
        

### **9.  Movie Recommendation Diversification (Complex Filtering + Optimization)**    
    
---    
    
#### Question    
       
        
*   Given a user's top 10 recommended movies (based on a simple collaborative filtering model - assume this is pre-computed), **re-rank these movies to increase genre diversity.**
*   Penalize movies that share genres with already-recommended movies.    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires reasoning about genre overlap and applying a penalty that discourages redundancy.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Pre-calculate genre sets for each movie.  Use a custom ranking function (UDF) that iteratively selects movies, penalizing based on genre overlap with previously selected movies.</font>         
        

### **10. Long-Tail Movie Recommendation (Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   **Identify "long-tail" movies (movies with few ratings but high average ratings).**
*   **For each user, recommend the top 3 long-tail movies that *do not* belong to genres the user has frequently rated.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Combines filtering based on rating count and average rating with a negative genre preference filter.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Pre-calculate user genre preferences.  Use efficient filtering and joining techniques to avoid unnecessary computations.</font>         
        

### **11.  Movie Release Year Influence (Windowing + Aggregation)**    
    
---    
    
#### Question    
       
        
*   **Analyze if the release year of a movie influences its ratings, considering user age.**
*   **Group users by age brackets (e.g., 18-25, 26-35, etc.) and movies by release year brackets.**
*   **Calculate the average rating for each user-age/movie-year combination.**
*   **Identify statistically significant correlations.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b> Requires grouping by two different time-based attributes and calculating correlations.</font>         
* <font color='lightgray'> <b>Optimization Hint:</b>  Pre-calculate age and release year brackets.  Use efficient grouping and aggregation operations.</font>         
        

### **12.  Genre Combination Preference (Aggregation + Complex Logic)**    
    
---    
    
#### Question    
       
        
*   **Identify combinations of *two* genres that, when present together in a movie, result in significantly higher average ratings than movies with only one of those genres.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires parsing the pipe-separated genres string, generating genre combinations, and comparing average ratings across different groups.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Use UDFs to efficiently parse genres and generate combinations.  Pre-calculate average ratings for single-genre movies.</font>         
        

### **13.  User Segmentation by Rating Patterns (Clustering - Conceptual, more like a discussion, *ignore this one in the first run*)**    

    I put the question here, as there's a separate module I am planning on clustering, MLLib and GraphX, this question whets your appetite.
---    
    
#### Question    
       
        
*   ***Describe* how you would use PySpark to segment users into distinct clusters based on their rating patterns (e.g., users who mostly give high ratings, users who are very critical, users with bimodal distributions, etc.).**
*   NOTE: You don't need to implement a full clustering algorithm, but outline the features you would extract and the PySpark operations you would use.    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b> Requires understanding of clustering concepts and how to translate them into PySpark feature engineering and (potentially) MLlib usage.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Consider features like average rating, rating variance, skewness, kurtosis, and percentile distributions.  Explain how you would use window functions, aggregations, and potentially UDFs to extract these features.</font>         
        

### **14.  Movie Similarity via Rating Vector Cosine Similarity (Aggregation + Optimization)**    
    
---    
    
#### Question    
       
        
*   **Calculate the *cosine similarity* between all pairs of movies based on their *user rating vectors*.**
*   Handle cases where users have rated one movie but not the other (impute a default rating or use a smoothing technique).
*   **Output the 10 most similar movies for each movie.**    
---
    
                
*    <font color='lightgray'> <b>Why this could be interesting:</b> High memory usage to create rating vectors, dealing with sparsity.</font>         
*    <font color='lightgray'> <b>Optimization Hint:</b> Instead of creating large vectors, calculate dot products and norms directly through joins and aggregations to save substantial memory.</font>         
        

### **15.  Impact of "Super-Raters" (Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   **Identify "super-raters" (users who have rated a significantly higher number of movies than average).**
*   Analyze the impact of removing their ratings on the overall average rating of movies.
*   **Are there specific genres or movies that are disproportionately affected?**
    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b> Requires identifying outliers in rating counts and then comparing aggregations with and without these users.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b> Use efficient filtering and aggregation techniques.  Calculate overall averages before and after removing super-raters.</font>         
        

### **16.  Predictive Rating Decay (Windowing + Regression - Conceptual, ummm...)**    
    ...again just embedding thought experiments here, these would be too large or costly to build as a part of this assessment
    
---    
    
#### Question    
       
        
*   ***Describe* how you would use PySpark to build a model that predicts how a movie's average rating will change over time (e.g., will it increase, decrease, or remain stable?).**
*   Consider factors like initial ratings, genre, and the ratings of similar movies.
*   **Outline the features and PySpark operations.**    
---
    
                
*    <font color='lightgray'> <b>Why this could be interesting:</b> Needs time-series analysis, feature engineering with genre, and similarity.</font>         
*    <font color='lightgray'> <b>Optimization Hint:</b> Create time-series features (average rating in different time windows).  Use genre and similarity (from previous questions) as features.  Explain how you would use window functions and potentially MLlib regression models.</font>         
        

### **17.  Transitive Genre Dislike Propagation (Graph-like + Optimization, Negative Preference)**    
    
---    
    
#### Question    
       
        
*   **Similar to Question 1, but propagate *dislike*.**
*   If a user consistently rates movies of Genre X poorly (<=2 stars), and they also rate movies of Genre Y poorly, then users who dislike Genre Y also indirectly dislike Genre X.
*   **Output the top 5 indirectly *disliked* genres for each user.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Mirrors the complexity of Question 1, but with the added nuance of handling negative preferences, which can be less intuitive.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b> Same optimization strategies as Question 1 apply, but carefully consider how to represent and propagate "dislike" efficiently.</font>         
        

### **18.  Genre Influence Over Time (Windowing + Correlation)**    
    
---    
    
#### Question    
       
        
*    Investigate the changing influence of genres on each other over time.
*    For each pair of genres, calculate the correlation between their average monthly ratings over a rolling 12-month window.
*    **Identify genre pairs with increasingly strong positive or negative correlations.**    
---
    
                
*    <font color='lightgray'> <b>Why this could be interesting:</b>  Requires nested windowing (monthly ratings within a 12-month rolling window) and correlation calculation.</font>         
*    <font color='lightgray'> <b>Optimization Hint:</b> Pre-calculate monthly average ratings for each genre. Use window functions to calculate the rolling 12-month correlation.</font>        
        

### **19.  User Rating Consistency Across Genres (Aggregation + Statistical Analysis)**    
    
---    
    
#### Question    
       
        
*   Analyze whether users maintain consistent rating behavior across different genres.
*   For each user and genre, calculate the standard deviation of their ratings.
*   **Identify users whose rating consistency varies significantly across genres.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b> Requires calculating per-user, per-genre statistics and then analyzing the variance of these statistics.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b> Use efficient grouping and aggregation operations. Calculate the standard deviation of standard deviations.</font>        
        

### **20.  Movie Recommendation "Serendipity" (Complex Filtering + Optimization)**    
    
---    
    
#### Question    
       
        
*   Design a metric to measure the **"serendipity" of a movie recommendation**.
*   A serendipitous recommendation is one that is both relevant (high predicted rating) and *unexpected* (not belonging to the user's frequently rated genres or similar to movies they've already seen).
*   **Implement** this metric.    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires combining relevance (predicted rating) with a measure of novelty or unexpectedness.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Use a combination of predicted ratings (from a pre-computed model), genre dissimilarity, and similarity to previously seen movies.</font>        
        

### **21.  Temporal Rating Bias (Windowing + Statistical Analysis)**    
    
---    
    
#### Question    
       
        
*   **Investigate if there's a temporal bias in user ratings (e.g., do users tend to rate higher or lower on weekends vs. weekdays, or during certain times of the year?).**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b> Requires extracting time-based features (day of week, month, etc.) and analyzing rating distributions across these features.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b> Use PySpark's date/time functions to extract relevant features.  Compare rating distributions using statistical tests.</font>        
        

### **22.  Influence of Early Ratings (Windowing + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Analyze the influence of a movie's *early* ratings (e.g., the first 100 ratings) on its long-term average rating.
*   **Do movies with high initial ratings tend to maintain high ratings, or is there a regression to the mean?**    
---
    
        
        
*    <font color='lightgray'> <b>Why this could be interesting:</b> Extracting first n ratings, calculate the moving averages, calculate correlations and handle edge cases.</font>        
*    <font color='lightgray'> <b>Optimization Hint:</b> Use window functions with a specific frame (first N rows) to calculate the early average rating.  Correlate this with the overall average rating.</font>        
        

### **23.  Genre Hybridity and Rating (Complex Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Define a measure of "genre hybridity" for a movie (e.g., the number of distinct genres it belongs to, or a more sophisticated measure based on genre co-occurrence).
*   **Investigate the relationship between genre hybridity and average rating.**    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires defining and calculating a complex metric based on the genre string, then correlating it with ratings.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b> Use UDFs to calculate genre hybridity.  Consider different weighting schemes for genres.</font>        
        

### **24.  User "Criticality" Evolution (Windowing + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Track how a user's "criticality" changes over time.
*   Define "criticality" as the difference between a user's average rating and the average rating of all users for the movies they've rated.
*   **Analyze if users tend to become more or less critical over time.**    
---
    
        
        
*    <font color='lightgray'> <b>Why this could be interesting:</b> Calculating user criticality relative to the average user.        
*    <font color='lightgray'> <b>Optimization Hint:</b> Use window functions to track a user's average rating and the overall average rating over time. Calculate the difference and analyze trends.</font>        
        

### **25.  Network Effects in Ratings (Graph-like + Optimization)**    
    
---    
    
#### Question    
       
        
*   Construct a user-movie bipartite graph.
*   **Investigate if there are "network effects" in ratings** – that is, if a user is connected to many users who have rated a movie highly, are they more likely to rate that movie highly as well, even after controlling for the movie's overall average rating?    
---
    
        
*   <font color='lightgray'> <b>Why this could be interesting:</b>  Requires constructing a bipartite graph representation and analyzing the influence of neighbors' ratings.</font>    
*   <font color='lightgray'> <b>Optimization Hint:</b>  Represent the data as an adjacency list (user -> [movies rated]) and use joins to find neighbors' ratings.  Calculate conditional probabilities to assess network effects. Use broadcasting of smaller graphs.</font>        

# Clear cache and stop the spark cluster

In [ ]:
# clear cache
spark.catalog.clearCache()

In [ ]:
# stop spark
spark.stop()